In [ ]:
%pip install mediapipe opencv-python numpy matplotlib scipy

## Initial SETUP

In [ ]:
import mediapipe as mp
import cv2
import numpy as np
import time
import threading
from collections import deque
import os
from scipy.signal import butter, filtfilt


BaseOptions = mp.tasks.BaseOptions
FaceLandmarker = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

model_path = "face_landmarker.task"

latest_result = None
result_lock = threading.Lock()

def store_result(result, output_image, timestamp_ms):
    global latest_result
    with result_lock:
        latest_result = result

options = FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=model_path),
    running_mode=VisionRunningMode.LIVE_STREAM,
    result_callback=store_result
)



## HELPER FUNCTIONS for CHROM 

In [ ]:
def chrom(rgb_signal):
    r = rgb_signal[:, 0].astype(np.float64)
    g = rgb_signal[:, 1].astype(np.float64)
    b = rgb_signal[:, 2].astype(np.float64)

    r_mean = np.mean(r) + 1e-6
    g_mean = np.mean(g) + 1e-6
    b_mean = np.mean(b) + 1e-6

    r_n = r / r_mean
    g_n = g / g_mean
    b_n = b / b_mean

    Xs = 3 * r_n - 2 * g_n
    Ys = 1.5 * r_n + g_n - 1.5 * b_n

    std_Xs = np.std(Xs) + 1e-6
    std_Ys = np.std(Ys) + 1e-6
    alpha = std_Xs / std_Ys

    pulse = Xs - alpha * Ys
    return pulse

def bandpass(signal, fps, low=0.7, high=3.5):
    nyq = fps / 2.0
    low_n = low / nyq
    high_n = high / nyq
    b, a = butter(4, [low_n, high_n], btype='band')
    return filtfilt(b, a, signal)

def estimate_hr(pulse_signal, fps):
    n = len(pulse_signal)

    n_padded = n * 4
    freqs = np.fft.rfftfreq(n_padded, d=1.0 / fps)
    fft_mag = np.abs(np.fft.rfft(pulse_signal, n=n_padded))

    valid = (freqs >= 0.7) & (freqs <= 3.5)
    if not np.any(valid):
        return 0

    peak_freq = freqs[valid][np.argmax(fft_mag[valid])]
    return peak_freq * 60

## FUNCTION to process either a webcam input or local video files

In [22]:
def process_video(is_livestream=True, video_folder_path=None, show_video=False):    
    regions = {
        "forehead": [109, 10, 338, 336, 9, 107],
        "left_cheek": [116, 111, 117, 118, 119, 120, 100, 142, 36, 205, 123],
        "right_cheek": [371, 329, 349, 348, 347, 346, 340, 345, 352, 425, 266]
    }

    gt_hr_array = None
    
    if is_livestream:
        cap = cv2.VideoCapture(0)
    else:
        video_path = f"{video_folder_path}/vid.avi"
        cap = cv2.VideoCapture(video_path)
        
        gt_file = f"{video_folder_path}/ground_truth.txt"
        
        if os.path.exists(gt_file):
            gt_data = np.loadtxt(gt_file)
            gt_hr_array = gt_data[1, :] 
            print(f"Loaded ground truth data: {len(gt_hr_array)} frames.")
        else:
            print(f"Warning: No ground truth file found at {gt_file}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    if not fps or fps == 0:
        fps = 30.0
    fps = float(fps)

    window_seconds = 7
    buffer_size = int(fps * window_seconds)
    rgb_buffer = deque(maxlen=buffer_size)

    bpm_display = 0
    last_hr_time = time.time()
    hr_update_interval = 1.5  

    start_time = time.time()
    time.sleep(1)
    
    frame_index = 0 
    pred_hr_array = [] 

    with FaceLandmarker.create_from_options(options) as landmarker:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            timestamp_ms = int((time.time() - start_time) * 1000)
            landmarker.detect_async(mp_image, timestamp_ms)

            with result_lock:
                current_result = latest_result

            if current_result and current_result.face_landmarks:
                landmarks = current_result.face_landmarks[0]
                h, w, _ = frame.shape

                for lm in landmarks:
                    x, y = int(lm.x * w), int(lm.y * h)
                    cv2.circle(frame, (x, y), 1, (0, 255, 0), -1)

                mask = np.zeros(frame.shape[:2], dtype=np.uint8)
                for region_name, ids in regions.items():
                    points = np.array([
                        (int(landmarks[i].x * w), int(landmarks[i].y * h))
                        for i in ids
                    ])
                    cv2.polylines(frame, [points], isClosed=True, color=(0, 0, 255), thickness=2)
                    cv2.fillPoly(mask, [points], 255)

                mean_bgr = cv2.mean(frame, mask=mask)[:3]
                B, G, R = mean_bgr
                rgb_buffer.append([R, G, B])

                now = time.time()
                if len(rgb_buffer) >= buffer_size and (now - last_hr_time) >= hr_update_interval:
                    last_hr_time = now

                    signal = np.array(rgb_buffer)   
                    pulse = chrom(signal)            
                    filtered = bandpass(pulse, fps)  
                    bpm_display = estimate_hr(filtered, fps)  

            color = (0, 255, 255) if bpm_display > 0 else (100, 100, 100)
            cv2.putText(frame, f"Pred HR: {bpm_display:.1f} BPM",
                        (30, 50), cv2.FONT_HERSHEY_SIMPLEX,
                        1.0, color, 2)
            
            cv2.putText(frame, f"Buffer: {len(rgb_buffer)}/{buffer_size}",
                        (30, 90), cv2.FONT_HERSHEY_SIMPLEX,
                        0.6, (200, 200, 200), 1)

            if gt_hr_array is not None and frame_index < len(gt_hr_array):
                current_gt_hr = float(gt_hr_array[frame_index]) 
                cv2.putText(frame, f"True HR: {current_gt_hr:.1f} BPM",
                            (30, 130), cv2.FONT_HERSHEY_SIMPLEX,
                            1.0, (0, 255, 0), 2)

            if show_video:
                cv2.imshow("rPPG", frame)
            
            pred_hr_array.append(bpm_display)
            
            frame_index += 1 

            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

    cap.release()
    cv2.destroyAllWindows()
    
    return np.array(pred_hr_array)

## FUNCTION to benchmark models against local video files (UBFC-rPPG dataset needs to be downloaded locally first, unfortunately I can't push those video files to github since they are too large)

In [ ]:
def run_benchmark(dataset_root, num_videos):
    all_maes = []
    all_mapes = []
    
    print(f"Starting frame-by-frame benchmark on {num_videos} videos...\n")
    print(f"{'Subject':<15} | {'Valid Frames':<12} | {'MAE (BPM)':<10} | {'MAPE (%)'}")
    print("-" * 65)
    
    subject_folders = [f for f in sorted(os.listdir(dataset_root)) if f.startswith("subject")][:num_videos]
    
    for subject in subject_folders:
        folder_path = os.path.join(dataset_root, subject)
        video_file = os.path.join(folder_path, "vid.avi")
        gt_file = os.path.join(folder_path, "ground_truth.txt")
        
        if not os.path.exists(video_file) or not os.path.exists(gt_file):
            continue
            
        pred_hr_array = process_video(is_livestream=False, video_folder_path=folder_path)
        
        gt_data = np.loadtxt(gt_file)
        true_hr_array = gt_data[1, :] 
        
        min_len = min(len(pred_hr_array), len(true_hr_array))
        preds = pred_hr_array[:min_len]
        gts = true_hr_array[:min_len]
        
        if min_len == 0:
            continue
            
        valid_indices = preds > 0
        if np.sum(valid_indices) == 0:
            print(f"{subject:<15} | Skipped (Video too short to fill buffer)")
            continue
            
        valid_preds = preds[valid_indices]
        valid_gts = gts[valid_indices]
            
        mae = np.mean(np.abs(valid_preds - valid_gts))
        mape = np.mean(np.abs((valid_gts - valid_preds) / valid_gts)) * 100
        
        all_maes.append(mae)
        all_mapes.append(mape)
        
        print(f"{subject:<15} | {len(valid_preds):<12} | {mae:<10.2f} | {mape:.2f}%")
        
    if len(all_maes) > 0:
        final_mae = np.mean(all_maes)
        final_mape = np.mean(all_mapes)
        
        print("\n" + "=" * 65)
        print("FINAL RESULTS")
        print("=" * 65)
        print(f"Total Mean Absolute Error (MAE):  {final_mae:.2f} BPM")
        print(f"Total Mean Abs Percentage Error:  {final_mape:.2f}%")
    else:
        print("\nNo valid videos were processed.")

In [ ]:
#run_benchmark(dataset_root='.', num_videos=3)

#commented out since this benchmark requires dataset videos to be downloaded locally

Starting frame-by-frame benchmark on 3 videos...

Subject         | Valid Frames | MAE (BPM)  | MAPE (%)
-----------------------------------------------------------------
Loaded ground truth data: 1547 frames.


W0000 00:00:1781286970.813025 17561226 face_landmarker_graph.cc:180] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
I0000 00:00:1781286970.826158 17561226 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M3 Pro
W0000 00:00:1781286970.829241 17561228 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1781286970.844897 17561233 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


subject1        | 1343         | 3.05       | 2.81%
Loaded ground truth data: 1989 frames.


W0000 00:00:1781286990.453093 17561829 face_landmarker_graph.cc:180] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
I0000 00:00:1781286990.461673 17561829 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M3 Pro
W0000 00:00:1781286990.464544 17561836 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1781286990.478583 17561840 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


subject12       | 1787         | 4.10       | 6.29%
Loaded ground truth data: 1958 frames.


W0000 00:00:1781287015.386303 17562533 face_landmarker_graph.cc:180] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
I0000 00:00:1781287015.389564 17562533 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M3 Pro
W0000 00:00:1781287015.391035 17562535 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1781287015.402633 17562536 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


subject23       | 1757         | 3.86       | 6.06%

FINAL RESULTS
Total Mean Absolute Error (MAE):  3.67 BPM
Total Mean Abs Percentage Error:  5.05%


In [ ]:
process_video(is_livestream=True, show_video=True)

#live demo with webcam feed

W0000 00:00:1781287041.517544 17564163 face_landmarker_graph.cc:180] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
I0000 00:00:1781287041.520450 17564163 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M3 Pro
W0000 00:00:1781287041.521687 17564166 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1781287041.529241 17564169 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


KeyboardInterrupt: 